In [ ]:
#%pip install rasterio

  Using cached rasterio-1.5.2-cp314-cp314-win_amd64.whl.metadata (8.9 kB)
  Using cached affine-3.0.1-py3-none-any.whl.metadata (4.5 kB)
Using cached rasterio-1.5.2-cp314-cp314-win_amd64.whl (31.5 MB)
Using cached affine-3.0.1-py3-none-any.whl (10 kB)

   ---------------------------------------- 0/3 [click]
   ---------------------------------------- 0/3 [click]
   ------------- -------------------------- 1/3 [affine]
   -------------------------- ------------- 2/3 [rasterio]
   -------------------------- ------------- 2/3 [rasterio]
   -------------------------- ------------- 2/3 [rasterio]
   -------------------------- ------------- 2/3 [rasterio]
   -------------------------- ------------- 2/3 [rasterio]
   -------------------------- ------------- 2/3 [rasterio]
   -------------------------- ------------- 2/3 [rasterio]
   -------------------------- ------------- 2/3 [rasterio]
   -------------------------- ------------- 2/3 [rasterio]
   -------------------------- ------------- 2/3

In [1]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping

In [2]:
import geopandas as gpd
plots_20_after_june_2017 = gpd.read_file("working\\obsevation_plots_with_year.gpkg")

In [3]:
plots_20_after_june_2017.head()

,Proovi / vaatluse kood,Sisestatud x L-EST97,Sisestatud y L-EST97,Seireaja algus,Elupaiga esinduslikkus,Elupaiga funktsioonide säilimine,Elupaiga üldine looduskaitseline väärtus,Elupaigatüüp (LD),Kasvukohatüüp (Paal 1997),Koosluse struktuuri säilimine,x,y,habitat,survey_date,source,habitat_code,plot_id,available_remote_sensing_year,geometry
0,V0000891786,6397087.0,648523.0,01.06.2017 00:00,A - väga hea,I - väga hea,A - väga kõrge väärtus,9080. Soostuvad ja soo-lehtmetsad,1.5.1.2. Jänesekapsa-kõdusoo kasvukohatüüp,NaN,648523.0,6397087.0,9080. Soostuvad ja soo-lehtmetsad,2017-06-01,field_plot,9080,0,2017,"POLYGON ((648543 6397087, 648542.904 6397085.0..."
1,V0000891788,6398159.0,648833.0,01.06.2017 00:00,B - hea,I - väga hea,A - väga kõrge väärtus,91D0*. Siirdesoo- ja rabametsad,1.4.2.1. Siirdesoometsa kasvukohatüüp,NaN,648833.0,6398159.0,91D0*. Siirdesoo- ja rabametsad,2017-06-01,field_plot,91D0*,1,2017,"POLYGON ((648853 6398159, 648852.904 6398157.0..."
2,V0000891736,6383581.0,651414.0,02.06.2017 00:00,C - arvestatav,II - hea,A - väga kõrge väärtus,9180*. Rusukallete ja jäärakute metsad,1.1.6.2. Naadi kasvukohatüüp,NaN,651414.0,6383581.0,9180*. Rusukallete ja jäärakute metsad,2017-06-02,field_plot,9180*,2,2017,"POLYGON ((651434 6383581, 651433.904 6383579.0..."
3,V0000891819,6458707.0,389146.0,14.06.2017 00:00,C - arvestatav,I - väga hea,B - kõrge väärtus,9050. Rohunditerikkad kuusikud,1.3.1.3. Sinihelmika kasvukohatüüp,NaN,389146.0,6458707.0,9050. Rohunditerikkad kuusikud,2017-06-14,field_plot,9050,3,2017,"POLYGON ((389166 6458707, 389165.904 6458705.0..."
4,V0000891817,6458791.0,389869.0,14.06.2017 00:00,A - väga hea,II - hea,A - väga kõrge väärtus,9060. Okasmetsad oosidel ja moreenkuhjatistel,1.1.5.1. Kassikäpa kasvukohatüüp,NaN,389869.0,6458791.0,9060. Okasmetsad oosidel ja moreenkuhjatistel,2017-06-14,field_plot,9060,4,2017,"POLYGON ((389889 6458791, 389888.904 6458789.0..."


In [4]:
plots_20_after_june_2017.shape[0]

1553

In [ ]:
#Defining the indices to be extracted from Sentinel imagery. Each entry specifies the satellite source, the corresponding data layer, and whether the dataset has median statistic for summer season. Alpha diversity is an exception as it does not have a median statistic in it's name, but could still represent median value.
indices = {
    "ndvi": ("sentinel2", "s2", True),
    "evi": ("sentinel2", "s2", True),
    "lai": ("sentinel2", "s2", True),
    "savi": ("sentinel2", "s2", True),
    "fvc": ("sentinel2", "s2", True),
    "gndvi": ("sentinel2", "s2", True),
    "ndwi": ("sentinel2", "s2", True),
    "bsi": ("sentinel2", "s2", True),
    "ndmi": ("sentinel2", "s2", True),
    "alpha_diversity": ("sentinel2", "s2", False),
    "rvi": ("sentinel1", "s1", True),
}

In [ ]:
#Read Sentinel imagery URLs for the specified indices and years. This function constructs the URL for each index based on its properties and the target year.
def cog_url(index, year):
    sensor, prefix, is_median = indices[index]
    statistic = "_median" if is_median else ""

    filename = (
        f"est_{prefix}_{index}{statistic}_"
        f"{year}-06-01_{year}-08-31_cog.tif"
    )

    return (
        f"https://s3.hpc.ut.ee/geokuup/estonia/"
        f"{sensor}/{index}/{year}/{filename}"
    )


In [7]:
plots_20_after_june_2017.dtypes

Proovi / vaatluse kood                                 str
Sisestatud x L-EST97                                   str
Sisestatud y L-EST97                                   str
Seireaja algus                                         str
Elupaiga esinduslikkus                                 str
Elupaiga funktsioonide säilimine                       str
Elupaiga üldine looduskaitseline väärtus               str
Elupaigatüüp (LD)                                      str
Kasvukohatüüp (Paal 1997)                              str
Koosluse struktuuri säilimine                          str
x                                                  float64
y                                                  float64
habitat                                                str
survey_date                                 datetime64[ms]
source                                                 str
habitat_code                                           str
plot_id                                              int

In [8]:
print(plots_20_after_june_2017.crs)
print(plots_20_after_june_2017.geom_type.unique())


EPSG:3301
<StringArray>
['Polygon']
Length: 1, dtype: str


In [ ]:
#Define the features dataframe for raster data extraction. This includes creating columns for each index to store mean, median, and number of pixels.
features = plots_20_after_june_2017.copy().reset_index(drop=True)

# Create columns for results
for index in indices:
    features[f"{index}_mean"] = np.nan
    features[f"{index}_median"] = np.nan
    features[f"{index}_n_pixels"] = 0


In [10]:
with rasterio.open(cog_url("ndvi", 2025)) as src:
    print("Data type:", src.dtypes[0])
    print("Scale:", src.scales[0])
    print("Offset:", src.offsets[0])
    print("Units:", src.units[0])
    print("Metadata:", src.tags(1))

Data type: int16
Scale: 1.0
Offset: 0.0
Units: None
Metadata: {}


In [11]:
with rasterio.open(cog_url("ndvi", 2025)) as src:
    data = src.read(1, masked=True, out_shape=(100, 100))
    values = data.compressed()

    print("Min:", values.min())
    print("Max:", values.max())
    print("Median:", np.median(values))
    print("Nodata:", src.nodata)

Min: -576
Max: 932
Median: 786.0
Nodata: -9999.0


In [ ]:
#Read and process Sentinel imagery for the specified indices and years. This section extracts raster data for each plot and stores the mean, median, and number of pixels for each index.
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY="2",GDAL_HTTP_RETRY_DELAY="1"
):
    # Process each year and index
    for year, plot_indices in features.dropna(subset=["available_remote_sensing_year"]).groupby("available_remote_sensing_year").groups.items():
        year = int(year)

        for index in indices:
            print(year, index)

            with rasterio.open(cog_url(index, year)) as src:

                # Match plot CRS to raster CRS
                subset = features.loc[plot_indices].to_crs(src.crs)

                for i, plot in subset.iterrows():

                    # Extract pixels within the 20 m plot
                    data, _ = mask(
                        src,
                        [mapping(plot.geometry)],
                        crop=True,
                        filled=False,
                        all_touched=True
                    )

                    # Remove nodata and NaN values
                    values = data[0].compressed()
                    values = values[np.isfinite(values)]

                    if len(values) == 0:
                        continue

        
                    values = values.astype(float) * 0.001

                    features.at[i, f"{index}_mean"] = values.mean()
                    features.at[i, f"{index}_median"] = np.median(values)
                    features.at[i, f"{index}_n_pixels"] = len(values)

2017 ndvi
2017 evi
2017 lai
2017 savi
2017 fvc
2017 gndvi
2017 ndwi
2017 bsi
2017 ndmi
2017 alpha_diversity
2017 rvi
2018 ndvi
2018 evi
2018 lai
2018 savi
2018 fvc
2018 gndvi
2018 ndwi
2018 bsi
2018 ndmi
2018 alpha_diversity
2018 rvi
2019 ndvi
2019 evi
2019 lai
2019 savi
2019 fvc
2019 gndvi
2019 ndwi
2019 bsi
2019 ndmi
2019 alpha_diversity
2019 rvi
2020 ndvi
2020 evi
2020 lai
2020 savi
2020 fvc
2020 gndvi
2020 ndwi
2020 bsi
2020 ndmi
2020 alpha_diversity
2020 rvi
2021 ndvi
2021 evi
2021 lai
2021 savi
2021 fvc
2021 gndvi
2021 ndwi
2021 bsi
2021 ndmi
2021 alpha_diversity
2021 rvi
2022 ndvi
2022 evi
2022 lai
2022 savi
2022 fvc
2022 gndvi
2022 ndwi
2022 bsi
2022 ndmi
2022 alpha_diversity
2022 rvi
2023 ndvi
2023 evi
2023 lai
2023 savi
2023 fvc
2023 gndvi
2023 ndwi
2023 bsi
2023 ndmi
2023 alpha_diversity
2023 rvi
2024 ndvi
2024 evi
2024 lai
2024 savi
2024 fvc
2024 gndvi
2024 ndwi
2024 bsi
2024 ndmi
2024 alpha_diversity
2024 rvi
2025 ndvi
2025 evi
2025 lai
2025 savi
2025 fvc
2025 gndvi
2025 n

In [ ]:
#Quick summary of extracted raster data for each index. This includes the number of valid and missing plots, as well as the median pixel count for each index.
summary = pd.DataFrame({
    index: {
        "valid_plots": features[f"{index}_mean"].notna().sum(),
        "missing_plots": features[f"{index}_mean"].isna().sum(),
        "median_pixel_count": features.loc[features[f"{index}_n_pixels"] > 0,f"{index}_n_pixels"].median()
    }
    for index in indices
}).T

display(summary)

print("Total plots:", len(features))


,valid_plots,missing_plots,median_pixel_count
ndvi,1553.0,0.0,22.0
evi,1553.0,0.0,22.0
lai,1553.0,0.0,22.0
savi,1553.0,0.0,22.0
fvc,1553.0,0.0,22.0
gndvi,1553.0,0.0,22.0
ndwi,1553.0,0.0,22.0
bsi,1553.0,0.0,22.0
ndmi,1553.0,0.0,22.0
alpha_diversity,1553.0,0.0,22.0


Total plots: 1553


In [14]:
features.dtypes

Proovi / vaatluse kood                                 str
Sisestatud x L-EST97                                   str
Sisestatud y L-EST97                                   str
Seireaja algus                                         str
Elupaiga esinduslikkus                                 str
Elupaiga funktsioonide säilimine                       str
Elupaiga üldine looduskaitseline väärtus               str
Elupaigatüüp (LD)                                      str
Kasvukohatüüp (Paal 1997)                              str
Koosluse struktuuri säilimine                          str
x                                                  float64
y                                                  float64
habitat                                                str
survey_date                                 datetime64[ms]
source                                                 str
habitat_code                                           str
plot_id                                              int

In [16]:
from pathlib import Path

print(Path.cwd())

c:\Users\kaursa40847\project


In [17]:
features.to_file("working/obsevation_plots_with_year_indices.gpkg", layer="obsevation_plots_with_year_indices", driver="GPKG")


In [18]:
features.head()

,Proovi / vaatluse kood,Sisestatud x L-EST97,Sisestatud y L-EST97,Seireaja algus,Elupaiga esinduslikkus,Elupaiga funktsioonide säilimine,Elupaiga üldine looduskaitseline väärtus,Elupaigatüüp (LD),Kasvukohatüüp (Paal 1997),Koosluse struktuuri säilimine,...,bsi_n_pixels,ndmi_mean,ndmi_median,ndmi_n_pixels,alpha_diversity_mean,alpha_diversity_median,alpha_diversity_n_pixels,rvi_mean,rvi_median,rvi_n_pixels
0,V0000891786,6397087.0,648523.0,01.06.2017 00:00,A - väga hea,I - väga hea,A - väga kõrge väärtus,9080. Soostuvad ja soo-lehtmetsad,1.5.1.2. Jänesekapsa-kõdusoo kasvukohatüüp,NaN,...,22,0.413182,0.4095,22,0.997182,0.956,22,0.881273,0.8725,22
1,V0000891788,6398159.0,648833.0,01.06.2017 00:00,B - hea,I - väga hea,A - väga kõrge väärtus,91D0*. Siirdesoo- ja rabametsad,1.4.2.1. Siirdesoometsa kasvukohatüüp,NaN,...,21,0.372810,0.3740,21,0.687810,0.726,21,1.110619,1.1130,21
2,V0000891736,6383581.0,651414.0,02.06.2017 00:00,C - arvestatav,II - hea,A - väga kõrge väärtus,9180*. Rusukallete ja jäärakute metsad,1.1.6.2. Naadi kasvukohatüüp,NaN,...,23,0.383130,0.3840,23,1.044913,0.949,23,0.775043,0.7690,23
3,V0000891819,6458707.0,389146.0,14.06.2017 00:00,C - arvestatav,I - väga hea,B - kõrge väärtus,9050. Rohunditerikkad kuusikud,1.3.1.3. Sinihelmika kasvukohatüüp,NaN,...,22,0.409909,0.4165,22,0.814909,0.797,22,1.078182,1.0615,22
4,V0000891817,6458791.0,389869.0,14.06.2017 00:00,A - väga hea,II - hea,A - väga kõrge väärtus,9060. Okasmetsad oosidel ja moreenkuhjatistel,1.1.5.1. Kassikäpa kasvukohatüüp,NaN,...,20,0.346750,0.3410,20,0.269300,0.267,20,0.841600,0.8455,20
